![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 9 -- Lab 2: Homework Overfitting and Regularization

Homework. Two experiments on the used-car listings of Day 08. First, on 80 Chevrolet Tahoes, you push a model from too simple to too flexible and watch the training and validation errors split, then repair the overfit with Ridge. Second, on the full 69-column table of Lab 1, you test whether Ridge and Lasso help when there are 5,000 rows, and read what Lasso throws away. The lesson is when regularization matters and how to tell.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Kaggle `turkibintalib/saudi-arabia-used-cars-dataset` (8,035 listings), downloaded with `kagglehub` as on Day 08.

---
# Setup

In [ ]:
# kagglehub, pandas, matplotlib and scikit-learn are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q kagglehub pandas matplotlib scikit-learn

In [ ]:
import os
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
path = kagglehub.dataset_download("turkibintalib/saudi-arabia-used-cars-dataset")
cars = pd.read_csv(os.path.join(path, "UsedCarsSA_Clean_EN.csv"))
cars = cars[cars["Price"] > 0].drop_duplicates().reset_index(drop=True)      # Day 08 Lab 1, Task 4
print(cars.shape)

In [ ]:
tahoe = cars[(cars["Type"] == "Tahoe") & (cars["Mileage"] <= 400000) & (cars["Year"] >= 2010) & (cars["Price"] < 400000)]
print(len(tahoe), "Chevrolet Tahoe listings")

---
# Part 1 -- From underfitting to overfitting

The *Two Ways to Be Wrong* and *The Gap Is the Alarm* slides. One feature (mileage), one target (price), and a model whose flexibility you control with the polynomial degree.

## Task 1: A small sample, split in half

Take `small = tahoe.sample(80, random_state=14)` and split it with `train_test_split(small, test_size=0.5, random_state=0)` into `tr` and `va`. Print both shapes (40 rows each). Draw a scatter plot of `Mileage` against `Price` for `tr` and `va` in two colours with a legend.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tr.shape == (40, 13) and va.shape == (40, 13), 'small: 80 rows with random_state=14, split 50/50 with random_state=0'
print('Task 1 passed')

## Task 2: A model whose flexibility is a dial

Write `poly_model(degree)` that returns `make_pipeline(StandardScaler(), PolynomialFeatures(degree), LinearRegression())`. Fit it with degree 1 on `tr[["Mileage"]]` and `tr["Price"]`, and print the MAE on the training rows and on the validation rows (expected 27,939 and 28,054: a line underfits, and both errors are high).

Syntax hint (the shape of the call, not the answer):

```python
pipe = make_pipeline(step1, step2, step3)      # fit and predict work on the whole pipeline
```

In [ ]:
# Your code here


## Task 3: Sweep the degree

For `degree` from 1 to 12, fit `poly_model(degree)` and record the training MAE and the validation MAE in a DataFrame `curve` with columns `degree`, `train_mae`, `val_mae`. Print it. Which degree has the smallest validation MAE (expected 3, about 23,171)? What is the validation MAE at degree 12 (expected about 57,081)?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert best_degree == 3 and curve['val_mae'].iloc[-1] > 50000, 'curve: one row per degree 1..12; the validation MAE is smallest at degree 3 and explodes at 12'
print('Task 3 passed')

## Task 4: Draw the alarm

Plot `train_mae` and `val_mae` against `degree` on one figure with a log-scaled y axis (`ax.set_yscale("log")`), a legend, a title and axis labels. The training curve only falls; the validation curve turns back up.

In [ ]:
# Your code here


---
# Part 2 -- Repair the overfit with Ridge

The *Why It Cures Overfitting* slide: keep the flexible degree-12 model, but make big weights expensive.

## Task 5: Ridge on the degree-12 features

Write `ridge_model(alpha)` returning `make_pipeline(StandardScaler(), PolynomialFeatures(12), StandardScaler(), Ridge(alpha=alpha))` (the second scaler puts the 12 powers on one scale so the penalty treats them equally). For `alpha` in `[0.001, 0.01, 0.1, 1, 10, 100, 1000]` fit on `tr` and print the validation MAE. Expected: about 34,972 at 0.001, a minimum of about 23,161 at alpha 1, and 37,752 at 1000. Store the alpha with the smallest validation MAE in `best_alpha`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert best_alpha == 1 and abs(results[1] - 23161) < 300, 'results: validation MAE of ridge_model(alpha) for each alpha; the minimum is at alpha = 1'
print('Task 5 passed')

---
# Part 3 -- Does regularization help on 5,000 rows?

The full table from Day 08's Lab 1 (the 69 columns after cleaning, log price as target) is rebuilt for you. Now the question is whether Ridge and Lasso change anything when the data is big.

In [ ]:
cars["Options_code"] = cars["Options"].map({"Standard": 0, "Semi Full": 1, "Full": 2})
cars["Gear_code"] = cars["Gear_Type"].map({"Manual": 0, "Automatic": 1})
numeric = ["Year", "Engine_Size", "Mileage", "Options_code", "Gear_code"]
X_all = pd.concat([cars[numeric], pd.get_dummies(cars[["Origin", "Fuel_Type"]], dtype=int), pd.get_dummies(cars["Make"], prefix="Make", dtype=int)], axis=1)
keep = (cars["Year"] >= 2000) & (cars["Mileage"] <= 1_000_000) & (cars["Price"] >= 5000)
X_full, y_log = X_all[keep], np.log(cars.loc[keep, "Price"])
X_train, X_val, y_train, y_val = train_test_split(X_full, y_log, test_size=0.2, random_state=42)
print(X_train.shape, X_val.shape)

## Task 6: A scoring helper on the price scale

Write `score(model)` that predicts `X_val`, turns the log predictions and `y_val` back to prices with `np.exp`, and returns `(MAE, RMSE, R2)` rounded. Score `make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)`. Expected: 20,218, 35,787 and 0.750, the same as the end of Day 08's Lab 1.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert score(linear)[0] == 20218, 'score: predict X_val, np.exp both sides, then MAE, RMSE, R2'
print('Task 6 passed')

## Task 7: Ridge on the big table

For `alpha` in `[0.01, 0.1, 1, 10, 100, 1000]` fit `make_pipeline(StandardScaler(), Ridge(alpha=alpha))` on the training rows and print `alpha` with `score(...)`. Expected: nothing changes until alpha 100 (MAE 20,362), and 1000 hurts (22,771). Write in a comment why the penalty does not help here when it halved the error in Part 2.

In [ ]:
# Your code here


## Task 8: Lasso throws columns away

For `alpha` in `[0.0001, 0.001, 0.01, 0.1]` fit `make_pipeline(StandardScaler(), Lasso(alpha=alpha, max_iter=20000))`, and print `alpha`, `score(...)` and the number of weights that are exactly 0 (`(m[-1].coef_ == 0).sum()`; `m[-1]` is the last step of the pipeline). Expected at 0.01: MAE about 21,153 with 24 zero weights. Store that model as `lasso`.

Syntax hint (the shape of the call, not the answer):

```python
last_step = pipe[-1]          # the fitted estimator inside a pipeline; its weights are last_step.coef_
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert int((lasso[-1].coef_ == 0).sum()) == 24, 'lasso: alpha=0.01, max_iter=20000, fitted on the training rows'
print('Task 8 passed')

## Task 9: What survived?

Put the Lasso weights in `pd.Series(lasso[-1].coef_, index=X_full.columns)`, keep the non-zero ones, sort them and print the six largest. Expected top: Year (0.469), Engine_Size (0.31), then Mercedes, Lexus, Options_code, Toyota. A model with 45 columns instead of 69 lost only about 900 SAR of MAE.

In [ ]:
# Your code here


## Task 10: Cross-validate the choice

One split can mislead. Run `cross_val_score` with `cv=5` and `scoring="neg_mean_absolute_error"` on `X_full, y_log` for `make_pipeline(StandardScaler(), Ridge(alpha=a))` with `a = 1` and `a = 100`, and print the negated means (MAE on the log scale). Expected about 0.2613 and 0.2630: the folds confirm that the penalty buys nothing here.

In [ ]:
# Your code here


---
## Task 11: Reflect

In Part 2 Ridge halved the validation error; in Part 3 it changed nothing. State the rule that explains both (think rows per weight). Why must `alpha` be chosen on validation rows or by cross-validation, never on the training error? Lasso dropped 24 columns for 900 SAR of MAE: give one reason a team might prefer the smaller model on Friday, and one reason not to.

*Your answers here*

---
## Conclusion

- **Underfitting to overfitting**: the polynomial degree moved the model from a line (both errors high) to a degree-12 curve (training error lowest, validation error worst).
- **The alarm**: the training curve only falls; the validation curve turns back up at degree 3.
- **Ridge repairs an overfit**: the same degree-12 model with alpha 1 matched the best hand-picked degree.
- **When it does not matter**: on 5,000 rows and 69 columns Ridge changed nothing, Lasso traded 24 columns for a small loss, and cross-validation confirmed both.

---

Made By **Sattam Altwaim**